# Demo 05a — Support Vector Machines

**Class 05 · Blocks 1-4** (slides: `slides/05_svm_neural_networks.md`)

Five live demos, each with more experiments than the slides show:

* **V1 — Margin and support vectors:** many lines separate the classes; the maximum-margin line is special.
* **V2 — Soft margin, the hinge loss and C:** the SVM as a loss plus a penalty, trained with `jax.grad`.
* **V3 — Kernels:** the same linear machine draws curved boundaries.
* **V4 — SVM in practice:** choosing C and gamma, scaling, cost, many classes, probabilities, unequal costs.
* **V5 — Support vector regression:** the tube and the epsilon-insensitive loss.

In [ ]:
import time
import warnings
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.datasets import make_blobs, make_circles, make_moons
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import HuberRegressor, LinearRegression, LogisticRegression, Perceptron, Ridge
from sklearn.metrics import brier_score_loss, confusion_matrix, log_loss
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR, LinearSVC

jax.config.update("jax_enable_x64", True)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
plt.rcParams["figure.figsize"] = (9, 4)
DATA = Path("../../datasets")


def split(X, y, seed=0, test=0.25):
    return train_test_split(X, y, test_size=test, stratify=y, random_state=seed)


pl.Config.set_tbl_rows(40)
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")


def table(rows, cols):
    """Print a list of rows as a polars table (hyperparameters such as C and gamma are written with `:g`)."""
    keep = {"C", "gamma", "epsilon", "learning rate"}
    rows = [tuple(f"{v:g}" if c in keep else v for c, v in zip(cols, r)) for r in rows]
    print(pl.DataFrame(rows, schema=cols, orient="row"))


def load_table(name, target):
    """Read `datasets/<name>.csv.zst` with polars: nulls -> median (numbers) or "missing" (text), text -> one-hot.

    Returns the feature matrix, the raw target column and the feature names.
    """
    df = pl.read_csv(DATA / f"{name}.csv.zst")
    X = df.drop(target)
    X = X.with_columns([pl.col(c).fill_null(pl.col(c).median()) for c, t in X.schema.items() if t.is_numeric()])
    cats = [c for c, t in X.schema.items() if t == pl.String]
    if cats:
        X = X.with_columns([pl.col(c).fill_null("missing") for c in cats]).to_dummies(columns=cats)
    return X.to_numpy().astype(float), df[target].to_numpy(), X.columns


def boundary(ax, model, X, y, title="", sv=None, levels=(-1, 0, 1)):
    gx, gy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 250), np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 250))
    z = model.decision_function(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, z > 0, alpha=0.12, cmap="coolwarm")
    ax.contour(gx, gy, z, levels=levels, colors="k", linestyles=["--", "-", "--"][: len(levels)], linewidths=1)
    ax.scatter(*X.T, c=y, cmap="coolwarm", s=18, edgecolor="k", linewidth=0.3)
    if sv is not None:
        ax.scatter(*sv.T, s=90, facecolor="none", edgecolor="k", linewidth=1.2)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

## V1 — Margin and support vectors

A **linear classifier** is a line (a hyperplane) $w^\top x + b = 0$. When the two classes can be separated, *many* lines do the
job. They all have zero training error, yet they are not equally good: the points closest to the line decide how much room a
new point has before it is misclassified. The **margin** is the distance from the line to the closest point;
the **support vector machine** chooses the line with the **largest** margin.

For a labelled point $(x_i, y_i)$ with $y_i \in \{-1, +1\}$:
the **functional margin** is $y_i (w^\top x_i + b)$ (positive when correct), and the **geometric margin** is
$y_i (w^\top x_i + b) / \lVert w\rVert$, the real distance in the units of the features.

**Example 1 of the slides.** Five points, two classes: the same data the slides use for the line, the distance, the margin and the
slack. Everything the slides compute by hand is checked here: the score, the distance and the margin of every point, the
maximum-margin line, the many lines of the perceptron (one for every order of the points) and logistic regression.

In [ ]:
X1 = np.array([[0, 0], [0, 1], [2, 0], [2, 1], [3, 1]], float)
y1 = np.array([-1, -1, 1, 1, 1])


def margin_table(w, b, X, y):
    f = X @ w + b
    return pl.DataFrame(
        {"point": [f"{i + 1}: {tuple(map(float, x))}" for i, x in enumerate(X)], "y": y, "f(x)": f, "m = y f": y * f,
         "distance": y * f / np.linalg.norm(w)})


w_h, b_h = np.array([1.0, 0.0]), -1.0
print("the line of the slides, w = (1, 0), b = -1:")
print(margin_table(w_h, b_h, X1, y1))
print("the same line with (2w, 2b): the functional margins double, the distances do not:")
print(margin_table(2 * w_h, 2 * b_h, X1, y1))
svm1 = SVC(kernel="linear", C=1e6).fit(X1, y1)
on_edge = [i + 1 for i in range(5) if abs(y1[i] * (X1[i] @ svm1.coef_[0] + svm1.intercept_[0]) - 1) < 1e-6]
print(f"SVC: w = {svm1.coef_[0].round(4)}, b = {svm1.intercept_[0]:.4f}, street {2 / np.linalg.norm(svm1.coef_[0]):.3f}")
print(f"points on the edge of the street (y f = 1): {on_edge}; the solver lists {sorted(int(i) + 1 for i in svm1.support_)} "
      "(with several points on the edge the dual multipliers are not unique)")

In [ ]:
import itertools


def perceptron_line(order):
    w, b = np.zeros(2), 0.0
    for _ in range(100):
        mistakes = 0
        for i in order:
            if y1[i] * (X1[i] @ w + b) <= 0:
                w, b, mistakes = w + y1[i] * X1[i], b + y1[i], mistakes + 1
        if mistakes == 0:
            break
    return w, b


lines = {}
for order in itertools.permutations(range(5)):
    w, b = perceptron_line(order)
    lines.setdefault((tuple(w), b), []).append(order)
rows = [(f"{w[0]:g} x1 + {w[1]:g} x2 + {b:g}", len(o), float(np.min(y1 * (X1 @ np.array(w) + b)) / np.linalg.norm(w))) for (w, b), o in lines.items()]
print(f"the perceptron over all {sum(len(o) for o in lines.values())} orders of the five points: {len(lines)} different lines")
print(pl.DataFrame(rows, schema=["line", "orders", "margin"], orient="row").sort("margin"))
rows = []
for C in (1, 10, 100, 10000):
    lr1 = LogisticRegression(C=C, max_iter=10000).fit(X1, y1)
    w, b = lr1.coef_[0], lr1.intercept_[0]
    rows.append((C, f"({w[0]:.3f}, {w[1]:.3f})", float(b), float(np.min(y1 * (X1 @ w + b)) / np.linalg.norm(w))))
print("logistic regression: a good line, not the widest (the SVM margin is 1.000):")
print(pl.DataFrame(rows, schema=["C", "w", "b", "margin"], orient="row"))

**The same idea with more points.** Two Gaussian blobs (60 points): the support vectors are the few points on the edge of the street.

In [ ]:
X, y01 = make_blobs(n_samples=60, centers=[(-2.0, -1.6), (2.0, 1.6)], cluster_std=0.9, random_state=3)
y = 2 * y01 - 1
svm = SVC(kernel="linear", C=1e6).fit(X, y)          # a huge C = a hard margin
w, b = svm.coef_[0], svm.intercept_[0]
margin = 1 / np.linalg.norm(w)
dist = y * (X @ w + b) / np.linalg.norm(w)
print(f"w = {w.round(3)}, b = {b:.3f}, ||w|| = {np.linalg.norm(w):.3f}")
print(f"geometric margin 1/||w|| = {margin:.3f}; smallest distance of a point to the line = {dist.min():.3f}")
print(f"support vectors: {len(svm.support_)} of {len(y)} points; their distances: {np.sort(dist)[:len(svm.support_)].round(3)}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
rng = np.random.default_rng(0)
xs = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 2)
for ax in axes[:2]:
    ax.scatter(*X.T, c=y01, cmap="coolwarm", s=22, edgecolor="k", linewidth=0.3)
# many separating lines: perceptron with random orders and starts
count = 0
margins = []
for seed in range(60):
    perm = np.random.default_rng(seed).permutation(len(y))
    p = Perceptron(max_iter=1000, tol=None, shuffle=False, eta0=1.0, random_state=seed).fit(X[perm] + rng.normal(0, 0.0, X.shape), y[perm])
    if (p.predict(X) == y).all():
        wp, bp = p.coef_[0], p.intercept_[0]
        margins.append(np.min(y * (X @ wp + bp)) / np.linalg.norm(wp))
        if count < 12:
            axes[0].plot(xs, -(wp[0] * xs + bp) / wp[1], color="tab:gray", lw=0.8)
            count += 1
axes[0].set_title(f"{count} of the many separating lines (perceptron)", fontsize=9)
boundary(axes[1], svm, X, y01, f"maximum margin = {2 * margin:.2f} wide", sv=svm.support_vectors_)
axes[2].hist(margins, bins=12, color="tab:gray", label=f"{len(margins)} perceptron lines")
axes[2].axvline(margin, color="tab:red", label="SVM")
axes[2].set(xlabel="geometric margin of the line", title="the SVM margin is the largest possible", xlim=(0, margin * 1.15))
axes[2].legend()
for ax in axes[:2]:
    ax.set_ylim(X[:, 1].min() - 1, X[:, 1].max() + 1)
    ax.set_xlim(X[:, 0].min() - 1, X[:, 0].max() + 1)
plt.tight_layout()
plt.show()

**Only the support vectors matter.** The solution is fixed by the few points that sit on the margin. Remove every other point
and retrain: the line is the same. Move a support vector and the line moves; move any other point (away from the margin) and
nothing changes.

In [ ]:
only_sv = SVC(kernel="linear", C=1e6).fit(X[svm.support_], y[svm.support_])
print("retrained on the support vectors only: |dw| =", np.abs(only_sv.coef_[0] - w).max().round(8), " |db| =", abs(only_sv.intercept_[0] - b).round(8))
far = np.setdiff1d(np.arange(len(y)), svm.support_)
shifted = X.copy()
shifted[far] += 0.3 * np.sign(X[far] @ w + b)[:, None] * w / np.linalg.norm(w)   # push the other points away from the line
moved = SVC(kernel="linear", C=1e6).fit(shifted, y)
print("pushing the non-support points further away: |dw| =", np.abs(moved.coef_[0] - w).max().round(8))
sv_move = X.copy()
sv_move[svm.support_[0]] += 0.5 * w / np.linalg.norm(w) * -y[svm.support_[0]]     # move one support vector towards the line
print("moving one support vector by 0.5: |dw| =", np.abs(SVC(kernel="linear", C=1e6).fit(sv_move, y).coef_[0] - w).max().round(3))

**A larger margin generalizes better.** Draw many datasets from the same two Gaussians, train a perceptron and an SVM, and
measure the error on a large fresh sample. The perceptron stops at *any* separating line; the SVM picks the best-placed one.
The gain is modest on average and large in the **worst case**: the margin protects against an unlucky line.

In [ ]:
def sample(n, seed, sep=1.5, std=1.0, shift=(4.0, -3.0)):
    Xs, ys = make_blobs(n_samples=n, centers=[(-sep, -sep), (sep, sep)], cluster_std=std, random_state=seed)
    return Xs + np.array(shift), 2 * ys - 1


Xbig, ybig = sample(20000, 999)
rows = []
for seed in range(200):
    Xs, ys = sample(14, seed)
    if len(np.unique(ys)) < 2:
        continue
    p = Perceptron(max_iter=5000, tol=None, random_state=seed).fit(Xs, ys)
    s = SVC(kernel="linear", C=1e6).fit(Xs, ys)
    if (p.predict(Xs) == ys).all() and (s.predict(Xs) == ys).all():
        rows.append(((p.predict(Xbig) != ybig).mean(), (s.predict(Xbig) != ybig).mean()))
rows = np.array(rows)
print(f"{len(rows)} separable training sets of 14 points; test error on 20000 fresh points")
for name, col in (("perceptron", 0), ("max-margin SVM", 1)):
    e = rows[:, col]
    print(f"{name:<15} mean {e.mean():.3f}   median {np.median(e):.3f}   90th percentile {np.percentile(e, 90):.3f}   worst {e.max():.3f}")
print(f"the SVM line has the smaller error in {np.mean(rows[:, 1] < rows[:, 0]):.0%} of the sets")
plt.boxplot([rows[:, 0], rows[:, 1]], tick_labels=["perceptron", "SVM"])
plt.ylabel("test error")
plt.title("any separating line (perceptron) versus the maximum-margin line")
plt.show()

## V2 — Soft margin, the hinge loss and C

Real classes overlap, so no line has zero error. The **soft margin** allows a point to be on the wrong side of its margin, at a
price: the **slack** $\xi_i = \max(0,\ 1 - y_i f(x_i))$ with $f(x) = w^\top x + b$. The SVM minimizes

$$\tfrac12 \lVert w\rVert^2 + C \sum_i \xi_i \quad\Longleftrightarrow\quad \frac{1}{2Cn}\lVert w\rVert^2 + \frac1n \sum_i \max\big(0,\ 1 - y_i f(x_i)\big).$$

The second form is **a loss (the hinge) plus a penalty ($L_2$)**, the recipe of Class 03; $\lambda = 1/(Cn)$.
Large $C$: errors are expensive, the margin is narrow. Small $C$: the margin is wide and tolerant.

**Slack on Example 1.** Add three points to Example 1: a blue one inside the street (a), a blue one exactly on the line (b) and
a red one on the wrong side (c). For the line $x_1 = 1$ of the slides the slack of each point is $\max(0, 1 - m)$; the soft-margin
SVM then trades the width of the street against the total slack, for every $C$.

In [ ]:
X1s = np.vstack([X1, [[1.5, 0.8], [1.0, 0.25], [1.4, 0.5]]])
y1s = np.r_[y1, 1, 1, -1]
m = y1s * (X1s @ np.array([1.0, 0.0]) - 1.0)
print(pl.DataFrame({"point": ["1", "2", "3", "4", "5", "a", "b", "c"], "m = y f": m, "slack": np.maximum(0, 1 - m)}))
print(f"total slack of the line x1 = 1: {np.maximum(0, 1 - m).sum():.2f}")
rows = []
for C in (0.1, 0.3, 1, 10, 100, 1000):
    s = SVC(kernel="linear", C=C, tol=1e-8).fit(X1s, y1s)
    w, b = s.coef_[0], s.intercept_[0]
    mm = y1s * (X1s @ w + b)
    rows.append((C, f"({w[0]:.3f}, {w[1]:.3f})", float(b), 2 / float(np.linalg.norm(w)), len(s.support_), float(np.maximum(0, 1 - mm).sum())))
print(pl.DataFrame(rows, schema=["C", "w", "b", "street", "support vectors", "total slack"], orient="row"))

In [ ]:
X, y01 = make_blobs(n_samples=200, centers=[(-1.0, -0.8), (1.0, 0.8)], cluster_std=1.3, random_state=1)
y = 2 * y01 - 1
Xte, yte01 = make_blobs(n_samples=4000, centers=[(-1.0, -0.8), (1.0, 0.8)], cluster_std=1.3, random_state=2)
yte = 2 * yte01 - 1
Cs = [0.001, 0.01, 0.1, 1, 10, 1000]
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
rows = []
for ax, C in zip(axes, Cs):
    m = SVC(kernel="linear", C=C).fit(X, y)
    wn = np.linalg.norm(m.coef_[0])
    rows.append((C, 2 / wn, len(m.support_), m.score(X, y), m.score(Xte, yte)))
    boundary(ax, m, X, y01, f"C = {C:g}", sv=m.support_vectors_)
plt.tight_layout()
plt.show()
table(rows, ["C", "margin width", "support vectors", "train acc", "test acc"])

The number of support vectors grows when $C$ is small (many points sit inside the wide margin). The training accuracy hardly
changes, but the test accuracy has a plateau: $C$ is a **hyperparameter** chosen on validation data (V4).

**The hinge loss is only one of several surrogates for the 0/1 loss.** All of them are functions of the margin
$m = y\,f(x)$:

In [ ]:
m = np.linspace(-2.5, 2.5, 400)
plt.plot(m, (m <= 0).astype(float), label="0/1 loss")
plt.plot(m, np.maximum(0, 1 - m), label="hinge (SVM)")
plt.plot(m, np.maximum(0, 1 - m) ** 2, label="squared hinge")
plt.plot(m, np.log1p(np.exp(-m)) / np.log(2), label="logistic (cross-entropy, base 2)")
plt.plot(m, np.maximum(0, -m), label="perceptron")
plt.plot(m, (1 - m) ** 2, label="squared error (1 - m)^2", ls=":")
plt.ylim(0, 3.5)
plt.xlabel("margin  y f(x)")
plt.ylabel("loss")
plt.legend()
plt.title("losses as functions of the margin")
plt.show()

### The SVM trained with `jax.grad`

The hinge is not differentiable at $m = 1$, but JAX returns a **subgradient** (0 where $m>1$, $-y\,x$ where $m<1$).
A few thousand Adam steps on the loss above reach the solution of the quadratic-programming solver (`SVC(kernel="linear")`).

In [ ]:
def hinge_loss(params, X_, y_, lam):
    margin_ = y_ * (X_ @ params["w"] + params["b"])
    return jnp.mean(jnp.maximum(0.0, 1.0 - margin_)) + 0.5 * lam * jnp.sum(params["w"] ** 2)


def adam(loss, params, *data, lr=0.05, steps=4000):
    vg = jax.jit(jax.value_and_grad(loss))
    m_ = jax.tree.map(jnp.zeros_like, params)
    v_ = jax.tree.map(jnp.zeros_like, params)
    hist = []
    for t in range(1, steps + 1):
        val, g = vg(params, *data)
        m_ = jax.tree.map(lambda a, gg: 0.9 * a + 0.1 * gg, m_, g)
        v_ = jax.tree.map(lambda a, gg: 0.999 * a + 0.001 * gg**2, v_, g)
        step = lr * 0.5 * (1 + np.cos(np.pi * (t - 1) / steps))
        params = jax.tree.map(lambda p, a, b_: p - step * (a / (1 - 0.9**t)) / (jnp.sqrt(b_ / (1 - 0.999**t)) + 1e-8), params, m_, v_)
        hist.append(float(val))
    return params, np.array(hist)


print(f"{'C':>6} {'|w_jax - w_svc|':>16} {'|b diff|':>9} {'same predictions':>17} {'objective jax':>14} {'objective svc':>14}")
for C in (0.1, 1, 10):
    lam = 1 / (C * len(y))
    p, h = adam(hinge_loss, {"w": jnp.zeros(2), "b": jnp.zeros(())}, X, y.astype(float), lam)
    ref = SVC(kernel="linear", C=C, tol=1e-10).fit(X, y)
    agree = np.mean(np.sign(Xte @ np.asarray(p["w"]) + float(p["b"])) == ref.predict(Xte))
    obj_ref = float(hinge_loss({"w": jnp.asarray(ref.coef_[0]), "b": jnp.asarray(ref.intercept_[0])}, X, y.astype(float), lam))
    print(f"{C:>6g} {np.abs(np.asarray(p['w']) - ref.coef_[0]).max():>16.4f} {abs(float(p['b']) - ref.intercept_[0]):>9.4f} {agree:>17.4f} {h[-1]:>14.5f} {obj_ref:>14.5f}")

**Support vectors are the points with margin $\le 1$** (slack $>0$ or exactly on the margin). Check it on the gradient solution:

In [ ]:
C = 1.0
lam = 1 / (C * len(y))
p, _ = adam(hinge_loss, {"w": jnp.zeros(2), "b": jnp.zeros(())}, X, y.astype(float), lam)
mg = y * (X @ np.asarray(p["w"]) + float(p["b"]))
ref = SVC(kernel="linear", C=C).fit(X, y)
print(f"points with margin <= 1 (gradient solution): {np.sum(mg <= 1 + 1e-3)}   support vectors of SVC: {len(ref.support_)}")
print(f"misclassified (margin < 0): {np.sum(mg < 0)}   inside the margin (0 <= margin < 1): {np.sum((mg >= 0) & (mg < 1))}")

### SVM against logistic regression, and scaling

Both are linear models with a different loss. Over many real datasets they behave alike; a difference comes from the loss
only at the margin. **Scaling matters** for both (the penalty $\lVert w\rVert^2$ treats the features alike).

In [ ]:
def load_xy(name, target):
    X, y, _ = load_table(name, target)
    return X, y


sets = {
    "breast_cancer": load_xy("breast_cancer", "target"),
    "spambase (log1p)": (lambda Xs, ys: (np.log1p(Xs), ys))(*load_xy("spambase", "spam")),
    "digits": (lambda Xs, ys: (Xs / 16, ys))(*load_xy("digits", "target")),
    "wine (quality >= 6)": (lambda Xs, ys: (Xs, (ys >= 6).astype(int)))(*load_xy("wine_quality", "quality")),
    "heart_disease": (lambda Xs, ys: (Xs, (ys == ">50_1").astype(int)))(*load_xy("heart_disease", "num")),
}
rows = []
for name, (Xd, yd) in sets.items():
    for label, model in (
        ("logistic", make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=5000))),
        ("linear SVM", make_pipeline(StandardScaler(), LinearSVC(C=1, max_iter=20000))),
        ("linear SVM, no scaling", LinearSVC(C=1, max_iter=20000)),
    ):
        accs = []
        for s_ in range(5):
            Xa, Xb_, ya, yb_ = split(Xd, yd, s_)
            accs.append(model.fit(Xa, ya).score(Xb_, yb_))
        rows.append((name, label, float(np.median(accs)), float(np.std(accs))))
table(rows, ["dataset", "model", "median test accuracy", "sd (5 splits)"])

**Choosing $C$ on validation data.** The validation curve of $C$ (log axis) for the breast-cancer data, with 10-fold
cross-validation, shows the plateau and the two failure modes.

In [ ]:
Xb, yb = sets["breast_cancer"]
Cgrid = np.logspace(-5, 3, 17)
skf = StratifiedKFold(10, shuffle=True, random_state=0)
tr_acc, va_acc, va_sd = [], [], []
for C in Cgrid:
    pipe = make_pipeline(StandardScaler(), LinearSVC(C=C, max_iter=50000))
    cv = [(pipe.fit(Xb[a], yb[a]).score(Xb[a], yb[a]), pipe.score(Xb[b_], yb[b_])) for a, b_ in skf.split(Xb, yb)]
    tr_acc.append(np.mean([c[0] for c in cv]))
    va_acc.append(np.mean([c[1] for c in cv]))
    va_sd.append(np.std([c[1] for c in cv]))
plt.semilogx(Cgrid, tr_acc, label="train")
plt.errorbar(Cgrid, va_acc, yerr=va_sd, label="validation (10-fold, mean ± sd)")
plt.xlabel("C (log)")
plt.ylabel("accuracy")
plt.legend()
plt.title("breast cancer, linear SVM")
plt.show()
best = int(np.argmax(va_acc))
print(f"best C = {Cgrid[best]:.3g} (validation accuracy {va_acc[best]:.3f}); C = {Cgrid[0]:.0e}: {va_acc[0]:.3f}; C = {Cgrid[-1]:.0e}: {va_acc[-1]:.3f}")

## V3 — Kernels

A line cannot separate a ring from its centre. The remedy of Class 03 was **new features** (polynomials): the same linear
model in a richer space. A **kernel** gives that space without ever building it.

*Why it works.* The SVM solution is $f(x) = \sum_i \alpha_i y_i\, x_i^\top x + b$: the training data enter **only through dot
products** $x_i^\top x$. Replace the dot product by $k(x_i, x) = \varphi(x_i)^\top\varphi(x)$ and the machine is linear in the
feature map $\varphi$, yet it never computes $\varphi$.

Start with the ring: the extra feature $r^2 = x_1^2 + x_2^2$ makes the problem linear.

In [ ]:
Xc, yc = make_circles(300, factor=0.4, noise=0.08, random_state=0)
Xc_te, yc_te = make_circles(2000, factor=0.4, noise=0.08, random_state=1)
lin = SVC(kernel="linear").fit(Xc, yc)
def r2(A):
    return np.c_[A, (A**2).sum(1)]


lift = SVC(kernel="linear", C=10).fit(r2(Xc), yc)
rbf = SVC(kernel="rbf", gamma=1.0, C=10).fit(Xc, yc)
print(f"linear on (x1, x2):                 test accuracy {lin.score(Xc_te, yc_te):.3f}")
print(f"linear on (x1, x2, x1^2 + x2^2):    test accuracy {lift.score(r2(Xc_te), yc_te):.3f}")
print(f"RBF kernel on (x1, x2):             test accuracy {rbf.score(Xc_te, yc_te):.3f}")
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
boundary(axes[0], lin, Xc, yc, "linear: 0.5, a line cannot", sv=lin.support_vectors_)
axes[1].scatter(Xc[:, 0], (Xc**2).sum(1), c=yc, cmap="coolwarm", s=14)
axes[1].set(xlabel="x1", ylabel="x1² + x2²", title="the extra feature: a line separates the classes")
boundary(axes[2], rbf, Xc, yc, "RBF kernel", sv=rbf.support_vectors_)
plt.tight_layout()
plt.show()

**A kernel is a dot product in a feature space.** For two inputs and the polynomial kernel $k(x, z) = (x^\top z)^2$, the
feature map is $\varphi(x) = (x_1^2,\ \sqrt2\,x_1 x_2,\ x_2^2)$. Check the identity with numbers:

In [ ]:
def phi(a):
    return np.array([a[0] ** 2, np.sqrt(2) * a[0] * a[1], a[1] ** 2])


a, b_ = np.array([1.0, 2.0]), np.array([3.0, -1.0])
print(f"(a . b)^2 = {(a @ b_) ** 2:.4f}     phi(a) . phi(b) = {phi(a) @ phi(b_):.4f}     phi(a) = {phi(a).round(3)}")
print("with d inputs and degree p the feature space has", "C(d+p, p) dimensions:", {(d, p): int(np.round(np.prod([(d + p - i) / (i + 1) for i in range(p)]))) for d in (2, 10, 64) for p in (2, 3)})

**The RBF (Gaussian) kernel** $k(x, z) = \exp(-\gamma\lVert x - z\rVert^2)$ is a **similarity**: 1 for equal points, near 0 for
far points. It corresponds to an *infinite-dimensional* $\varphi$. The Gram matrix $K_{ij} = k(x_i, x_j)$ contains all the
similarities; written with `jax.numpy` and broadcasting:

In [ ]:
def rbf_kernel(A, B, gamma):
    d2 = jnp.sum(A**2, 1)[:, None] + jnp.sum(B**2, 1)[None, :] - 2 * A @ B.T      # squared distances, all pairs
    return jnp.exp(-gamma * d2)


order = np.argsort(yc)
K = np.asarray(rbf_kernel(jnp.asarray(Xc[order]), jnp.asarray(Xc[order]), 2.0))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im = axes[0].imshow(K, cmap="viridis")
axes[0].set_title("Gram matrix, points sorted by class")
plt.colorbar(im, ax=axes[0])
for g in (0.1, 1, 10, 100):
    dd = np.linspace(0, 3, 100)
    axes[1].plot(dd, np.exp(-g * dd**2), label=f"gamma = {g:g}")
axes[1].set(xlabel="distance ||x - z||", ylabel="k(x, z)", title="similarity decays with distance")
axes[1].legend()
plt.tight_layout()
plt.show()
print("max |our Gram matrix - sklearn rbf_kernel| =", float(np.abs(K - __import__("sklearn.metrics.pairwise", fromlist=["rbf_kernel"]).rbf_kernel(Xc[order], gamma=2.0)).max()))

**The Gram matrix of Example 1** (the heat map of the slides): the two groups show up as blocks.

In [ ]:
K1 = np.exp(-0.5 * ((X1[:, None] - X1[None]) ** 2).sum(-1))
print("Gram matrix of Example 1, RBF kernel with gamma = 0.5 (rows and columns are the points 1-5):")
print(np.round(K1, 2))

### The kernel SVM with `jax.grad` (the representer form)

The optimal $f$ is a weighted sum of similarities to the training points, $f(x) = \sum_j \beta_j\, k(x_j, x) + b$.
The hinge loss plus the penalty $\tfrac{\lambda}{2}\beta^\top K\beta$ is trained directly for $\beta$ with `jax.grad`,
with no quadratic-programming solver.

In [ ]:
def kernel_svm_loss(params, K_, y_, lam):
    f = K_ @ params["beta"] + params["b"]
    return jnp.mean(jnp.maximum(0.0, 1.0 - y_ * f)) + 0.5 * lam * params["beta"] @ K_ @ params["beta"]


ycs = 2.0 * yc - 1
C, gamma = 10.0, 1.0
Ktr = rbf_kernel(jnp.asarray(Xc), jnp.asarray(Xc), gamma)
Kte = rbf_kernel(jnp.asarray(Xc_te), jnp.asarray(Xc), gamma)
p, h = adam(kernel_svm_loss, {"beta": jnp.zeros(len(yc)), "b": jnp.zeros(())}, Ktr, jnp.asarray(ycs), 1 / (C * len(yc)), lr=0.02, steps=6000)
ours = np.asarray(Kte @ p["beta"] + p["b"])
ref = SVC(kernel="rbf", gamma=gamma, C=C).fit(Xc, yc)
print(f"test accuracy: ours {np.mean((ours > 0) == yc_te):.3f}, SVC {ref.score(Xc_te, yc_te):.3f}")
print(f"same prediction on {np.mean((ours > 0) == ref.predict(Xc_te)):.1%} of the test points; support vectors: SVC {len(ref.support_)}, ours (|beta| > 1e-3) {int(np.sum(np.abs(np.asarray(p['beta'])) > 1e-3))}")
print(f"correlation of the two decision functions: {np.corrcoef(ours, ref.decision_function(Xc_te))[0, 1]:.4f}")

### Gamma: how local is the similarity?

Small $\gamma$ = wide bells = a smooth, almost linear boundary (underfit). Large $\gamma$ = narrow bells = each point
defends its own small region (overfit). $\gamma$ is a hyperparameter, like $C$.

In [ ]:
Xm, ym = make_moons(250, noise=0.3, random_state=0)
Xm_te, ym_te = make_moons(4000, noise=0.3, random_state=1)
gammas = [0.01, 0.1, 1, 10, 100, 1000]
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
rows = []
for ax, g in zip(axes, gammas):
    mdl = SVC(kernel="rbf", gamma=g, C=10).fit(Xm, ym)
    boundary(ax, mdl, Xm, ym, f"gamma = {g:g}", sv=mdl.support_vectors_, levels=(0,))
    rows.append((g, len(mdl.support_), mdl.score(Xm, ym), mdl.score(Xm_te, ym_te)))
plt.tight_layout()
plt.show()
table(rows, ["gamma", "support vectors", "train acc", "test acc"])

**Which kernel?** The same data, five kernels. On `moons` and `circles` the linear kernel fails; on the breast-cancer
measurements (30 features, almost linearly separable) it is already hard to beat.

In [ ]:
datasets = {
    "moons": (Xm, ym, Xm_te, ym_te),
    "circles": (Xc, yc, Xc_te, yc_te),
}
Xb, yb = sets["breast_cancer"]
a_, b2_, c_, d_ = split(Xb, yb, 0)
sc = StandardScaler().fit(a_)
datasets["breast_cancer"] = (sc.transform(a_), c_, sc.transform(b2_), d_)
kernels = {
    "linear": dict(kernel="linear"),
    "poly d=2": dict(kernel="poly", degree=2, coef0=1, gamma="scale"),
    "poly d=3": dict(kernel="poly", degree=3, coef0=1, gamma="scale"),
    "poly d=7": dict(kernel="poly", degree=7, coef0=1, gamma="scale"),
    "rbf": dict(kernel="rbf", gamma="scale"),
    "sigmoid": dict(kernel="sigmoid", gamma="scale"),
}
rows = [(kname, *[SVC(C=1, **kw).fit(Xa, ya).score(Xe, ye) for Xa, ya, Xe, ye in datasets.values()]) for kname, kw in kernels.items()]
table(rows, ["kernel", *datasets])

**Scaling again.** The RBF kernel is a function of the Euclidean distance: a feature with a large range decides the distance
and the others are ignored. The breast-cancer features range from 0.05 (smoothness) to 2500 (area).

In [ ]:
Xb_tr, Xb_te, yb_tr, yb_te = split(Xb, yb, 0)
print("feature ranges (max - min):", dict(zip(["mean_radius", "mean_smoothness", "mean_area"], [round(float(np.ptp(Xb[:, j])), 2) for j in (0, 4, 3)])))
for label, (A, B) in {"raw features": (Xb_tr, Xb_te), "standardized": (sc.transform(Xb_tr), sc.transform(Xb_te))}.items():
    mdl = SVC(kernel="rbf", gamma="scale", C=1).fit(A, yb_tr)
    print(f"RBF SVM on {label:<13} test accuracy {mdl.score(B, yb_te):.3f}  (gamma = 'scale' = {1 / (A.shape[1] * A.var()):.2e})")

The gradient solution has 298 non-zero $\beta_j$, the QP solver 12: the solver's *dual* variables are exactly zero
outside the support vectors, while gradient descent only makes the other $\beta_j$ small. The **predictions** agree.

## V4 — The SVM in practice

Four decisions: scaling (done), the kernel, the pair ($C$, $\gamma$), and how to report the result honestly.

**The grid.** $C$ and $\gamma$ act on different *scales*, so the grid is **logarithmic**. A heat map of the cross-validated
accuracy on the handwritten digits (1797 images, 64 pixels, 10 classes):

In [ ]:
Xd, yd = sets["digits"]
Xd_tr, Xd_te, yd_tr, yd_te = split(Xd, yd, 0)
Cs = 10.0 ** np.arange(-2, 5)
gs = 10.0 ** np.arange(-5, 1)
t0 = time.time()
grid = GridSearchCV(SVC(kernel="rbf"), {"C": Cs, "gamma": gs}, cv=StratifiedKFold(5, shuffle=True, random_state=0), n_jobs=-1).fit(Xd_tr, yd_tr)
score = grid.cv_results_["mean_test_score"].reshape(len(Cs), len(gs))
fig, ax = plt.subplots(figsize=(7, 4.5))
im = ax.imshow(score, cmap="viridis", origin="lower", vmin=0.1)
ax.set_xticks(range(len(gs)), [f"1e{int(np.log10(g))}" for g in gs])
ax.set_yticks(range(len(Cs)), [f"1e{int(np.log10(c))}" for c in Cs])
ax.set(xlabel="gamma (log)", ylabel="C (log)", title="cross-validated accuracy, digits")
for i in range(len(Cs)):
    for j in range(len(gs)):
        ax.text(j, i, f"{score[i, j]:.2f}", ha="center", va="center", color="w", fontsize=7)
plt.colorbar(im)
plt.show()
print(f"grid of {len(Cs) * len(gs)} pairs x 5 folds in {time.time() - t0:.1f} s; best {grid.best_params_}, CV accuracy {grid.best_score_:.3f}")
print(f"test accuracy of the refitted best model: {grid.score(Xd_te, yd_te):.3f}; default SVC(): {SVC().fit(Xd_tr, yd_tr).score(Xd_te, yd_te):.3f}; gamma='scale' = {1 / (64 * Xd_tr.var()):.4f}")

**The optimism of the best grid cell.** The best cross-validation score is the maximum of 42 noisy numbers, so it is
*biased upwards*. Nested cross-validation (Class 01) repeats the whole search inside each outer fold and scores the outer
test part only. On a *small* dataset the difference is visible (breast cancer, 569 examples, 20 outer repetitions):

In [ ]:
Xb, yb = sets["breast_cancer"]
small_grid = {"svc__C": 10.0 ** np.arange(-2, 4), "svc__gamma": 10.0 ** np.arange(-5, 0)}
pipe = make_pipeline(StandardScaler(), SVC())
naive, nested = [], []
for rep in range(10):
    outer = StratifiedKFold(5, shuffle=True, random_state=rep)
    inner = StratifiedKFold(5, shuffle=True, random_state=100 + rep)
    g = GridSearchCV(pipe, small_grid, cv=inner, n_jobs=-1).fit(Xb, yb)
    naive.append(g.best_score_)
    nested.append(cross_val_score(GridSearchCV(pipe, small_grid, cv=inner, n_jobs=-1), Xb, yb, cv=outer).mean())
print(f"best-cell CV accuracy (10 repetitions): {np.mean(naive):.4f} ± {np.std(naive):.4f}")
print(f"nested CV accuracy   (10 repetitions): {np.mean(nested):.4f} ± {np.std(nested):.4f}   optimism = {np.mean(naive) - np.mean(nested):+.4f}")

**The cost.** Training an SVM solves a problem with $n$ variables and a Gram matrix of $n^2$ entries: the time grows between
$n^2$ and $n^3$. A linear model grows like $n$. Measured on the spam data (57 features):

In [ ]:
Xs_, ys_ = sets["spambase (log1p)"]
Xs_ = StandardScaler().fit_transform(Xs_)
rng = np.random.default_rng(0)
rows = []
ns, t_rbf = [], []
for n in (250, 500, 1000, 2000, 3500):
    idx = rng.choice(len(ys_), n, replace=False)
    t = time.time()
    s = SVC(kernel="rbf").fit(Xs_[idx], ys_[idx])
    t1 = time.time() - t
    t = time.time()
    LinearSVC(max_iter=20000).fit(Xs_[idx], ys_[idx])
    t2 = time.time() - t
    t = time.time()
    LogisticRegression(max_iter=2000).fit(Xs_[idx], ys_[idx])
    t3 = time.time() - t
    ns.append(n)
    t_rbf.append(t1)
    rows.append((n, t1, len(s.support_), t2, t3))
table(rows, ["n", "RBF SVC fit (s)", "support vectors", "LinearSVC fit (s)", "logistic fit (s)"])
print(f"log-log slope of the RBF fit time: {np.polyfit(np.log(ns), np.log(t_rbf), 1)[0]:.2f}")
print("the Gram matrix of 100 000 examples in float64 needs", f"{100_000**2 * 8 / 1e9:.0f} GB")

**More than two classes.** The SVM is binary. *One-vs-rest* trains $K$ models (a class against all others), *one-vs-one*
trains $K(K-1)/2$ models on pairs and votes. scikit-learn's `SVC` uses one-vs-one internally; the number of models matters for
the cost and the shape of `decision_function`.

In [ ]:
from sklearn.multiclass import OneVsRestClassifier

best = {k: float(v) for k, v in grid.best_params_.items()}
ovo = SVC(kernel="rbf", decision_function_shape="ovo", **best).fit(Xd_tr, yd_tr)
ovr = OneVsRestClassifier(SVC(kernel="rbf", **best)).fit(Xd_tr, yd_tr)
print(f"one-vs-one:  {len(ovo.support_vectors_)} support vectors; decision_function shape {ovo.decision_function(Xd_te).shape}; test accuracy {ovo.score(Xd_te, yd_te):.3f}")
print(f"one-vs-rest: {len(ovr.estimators_)} models; decision_function shape {ovr.decision_function(Xd_te).shape}; test accuracy {ovr.score(Xd_te, yd_te):.3f}")
cm = confusion_matrix(yd_te, ovo.predict(Xd_te))
off = cm - np.diag(np.diag(cm))
i, j = np.unravel_index(off.argmax(), off.shape)
print(f"most frequent confusion: true {i} predicted as {j}, {off[i, j]} times; total errors {off.sum()} of {len(yd_te)}")
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(np.log1p(cm), cmap="Blues")
ax.set(xlabel="predicted", ylabel="true", title="digits, RBF SVM (log colour scale)")
plt.show()

**Probabilities.** The SVM outputs a **score** $f(x)$ (a signed distance), not a probability. Platt scaling (Class 04) fits a
sigmoid $\sigma(a f + b)$ on held-out scores; `CalibratedClassifierCV(SVC(), method="sigmoid")` does it with cross-validation.
Compare the probability quality (log-loss, Brier) with logistic regression on the spam data.

In [ ]:
Xs_tr, Xs_te, ys_tr, ys_te = split(Xs_, ys_, 0)
svc = CalibratedClassifierCV(SVC(kernel="rbf", C=1), method="sigmoid", cv=5, ensemble=False).fit(Xs_tr, ys_tr)
lr = LogisticRegression(C=1, max_iter=5000).fit(Xs_tr, ys_tr)
raw = SVC(kernel="rbf", C=1).fit(Xs_tr, ys_tr).decision_function(Xs_te)
naive = 1 / (1 + np.exp(-raw))
print(f"{'model':<26} {'accuracy':>9} {'log-loss':>9} {'Brier':>7}")
for name, p_, pred in (("SVM, sigmoid(raw score)", naive, raw > 0), ("SVM, Platt scaling", svc.predict_proba(Xs_te)[:, 1], svc.predict(Xs_te)), ("logistic regression", lr.predict_proba(Xs_te)[:, 1], lr.predict(Xs_te))):
    print(f"{name:<26} {np.mean(pred == ys_te):>9.3f} {log_loss(ys_te, np.clip(p_, 1e-12, 1 - 1e-12)):>9.3f} {brier_score_loss(ys_te, p_):>7.3f}")
fig, ax = plt.subplots(figsize=(4.5, 4))
for name, p_ in (("SVM sigmoid(raw)", naive), ("SVM Platt", svc.predict_proba(Xs_te)[:, 1]), ("logistic", lr.predict_proba(Xs_te)[:, 1])):
    fp, mp = calibration_curve(ys_te, p_, n_bins=8)
    ax.plot(mp, fp, "o-", label=name)
ax.plot([0, 1], [0, 1], "k:")
ax.set(xlabel="predicted probability", ylabel="observed frequency", title="reliability")
ax.legend()
plt.show()

**Unequal costs.** In the German credit data a bad credit accepted costs 5 times a good credit refused. The SVM minimizes
errors; to minimize *cost*, give the rare and expensive class a larger $C$ (`class_weight`), exactly as a threshold does for a
probability (Class 04).

In [ ]:
Xg, cls, _ = load_table("german_credit", "class")
yg = (cls == "bad").astype(int)          # 1 = bad credit (the expensive error)
Xg_tr, Xg_te, yg_tr, yg_te = split(Xg, yg, 0)
sc = StandardScaler().fit(Xg_tr)
Xg_tr, Xg_te = sc.transform(Xg_tr), sc.transform(Xg_te)
print(f"{Xg.shape[1]} features after one-hot encoding; {yg.mean():.0%} bad credits")
rows = []
for label, kw in (("none", {}), ("bad = 5", {"class_weight": {0: 1, 1: 5}}), ("balanced", {"class_weight": "balanced"})):
    mdl = SVC(kernel="rbf", C=1, **kw).fit(Xg_tr, yg_tr)
    cm = confusion_matrix(yg_te, mdl.predict(Xg_te))
    rows.append((label, mdl.score(Xg_te, yg_te), int(cm[1, 0]), int(cm[0, 1]), int(5 * cm[1, 0] + cm[0, 1])))
table(rows, ["class_weight", "accuracy", "missed bad", "refused good", "cost (5:1)"])

## V5 — Support vector regression

For a numeric target the SVM idea becomes a **tube**: errors smaller than $\varepsilon$ cost nothing, larger ones cost
linearly. The **$\varepsilon$-insensitive loss** is $\ell_\varepsilon(r) = \max(0,\ |r| - \varepsilon)$ for the residual
$r = y - f(x)$. Points inside the tube do not influence the model: the support vectors are the points on or outside
the tube. $\varepsilon$ plays the role of the noise level and $C$ that of the penalty, as before.

In [ ]:
rng = np.random.default_rng(0)
xs = np.sort(rng.uniform(0, 6, 80))
ys = np.sin(xs) + rng.normal(0, 0.25, xs.size)
xg = np.linspace(0, 6, 300)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.4), sharey=True)
rows = []
for ax, eps in zip(axes, (0.0, 0.1, 0.3, 0.6)):
    mdl = SVR(kernel="rbf", C=10, epsilon=eps, gamma=1).fit(xs[:, None], ys)
    f = mdl.predict(xg[:, None])
    ax.fill_between(xg, f - eps, f + eps, alpha=0.25, color="tab:orange")
    ax.plot(xg, f, color="tab:red")
    ax.scatter(xs, ys, s=14, c="tab:blue")
    ax.scatter(xs[mdl.support_], ys[mdl.support_], s=60, facecolor="none", edgecolor="k")
    ax.set_title(f"epsilon = {eps}", fontsize=9)
    rows.append((eps, len(mdl.support_), len(mdl.support_) / len(xs), float(np.sqrt(np.mean((mdl.predict(xs[:, None]) - ys) ** 2)))))
plt.tight_layout()
plt.show()
table(rows, ["epsilon", "support vectors", "fraction", "train RMSE"])

**The loss, and outliers.** Squared error grows without limit, so a single outlier pulls the fit. The $\varepsilon$-insensitive
loss grows linearly (like the absolute error), so large residuals have a bounded influence.

In [ ]:
r = np.linspace(-3, 3, 400)
plt.plot(r, r**2, label="squared error")
plt.plot(r, np.abs(r), label="absolute error")
plt.plot(r, np.maximum(0, np.abs(r) - 0.5), label="epsilon-insensitive (eps = 0.5)")
plt.ylim(0, 4)
plt.xlabel("residual y - f(x)")
plt.legend()
plt.show()

x1 = np.linspace(0, 10, 40)
y1 = 2 * x1 + 1 + rng.normal(0, 1, 40)
y1[[5, 20, 33]] += [25, 30, 28]                        # three gross outliers
print(f"{'model':<24} {'slope':>7} {'intercept':>10}   (true line: slope 2, intercept 1)")
for name, mdl in (("least squares", LinearRegression()), ("SVR linear, eps = 0.5", SVR(kernel="linear", C=10, epsilon=0.5)), ("Huber regression", HuberRegressor())):
    mdl.fit(x1[:, None], y1)
    coef = mdl.coef_.ravel()[0]
    print(f"{name:<24} {coef:>7.2f} {float(np.ravel(mdl.intercept_)[0]):>10.2f}")

**Real data: concrete strength** (1030 mixtures, 8 numeric features, target in MPa). Three models, 5-fold cross-validation,
the preprocessing inside the pipeline. The target is scaled too: $\varepsilon$ is expressed in the units of the target.

In [ ]:
Xc_, yc_ = load_xy("concrete", "strength")
cv = StratifiedKFold(5, shuffle=True, random_state=0)
bins = np.digitize(yc_, np.quantile(yc_, [0.25, 0.5, 0.75]))
def cv_rmse(model):
    out = []
    for a, b_ in cv.split(Xc_, bins):
        mu, sd = yc_[a].mean(), yc_[a].std()
        model.fit(Xc_[a], (yc_[a] - mu) / sd)
        out.append(np.sqrt(np.mean((model.predict(Xc_[b_]) * sd + mu - yc_[b_]) ** 2)))
    return np.mean(out), np.std(out)


models = {
    "mean (baseline)": make_pipeline(StandardScaler(), Ridge(alpha=1e9)),
    "ridge": make_pipeline(StandardScaler(), Ridge(alpha=1)),
    "SVR linear": make_pipeline(StandardScaler(), SVR(kernel="linear", C=1, epsilon=0.1)),
    "SVR rbf (default)": make_pipeline(StandardScaler(), SVR(kernel="rbf")),
    "SVR rbf (C=30, gamma=0.2, eps=0.05)": make_pipeline(StandardScaler(), SVR(kernel="rbf", C=30, gamma=0.2, epsilon=0.05)),
}
rows = [(name, *cv_rmse(mdl)) for name, mdl in models.items()]
table(rows, ["model (concrete)", "RMSE (MPa)", "sd over folds"])

**Other datasets, same protocol.** Abalone (age in rings from shell measurements; one categorical feature `sex`),
auto-mpg (fuel consumption; 6 missing values of `horsepower`) and the diabetes data (10 standardized features, a very noisy
target). `load_table` fills the nulls and one-hot encodes the text columns with polars. The metric is the RMSE of
5-fold cross-validation, relative to the baseline that always predicts the mean.

In [ ]:
rows = []
for dname, target in (("abalone", "rings"), ("auto_mpg", "mpg"), ("diabetes", "target"), ("wine_quality", "quality")):
    Xr, yr, _ = load_table(dname, target)
    yr = yr.astype(float)
    bins_ = np.digitize(yr, np.quantile(yr, [0.25, 0.5, 0.75]))
    res = {}
    for label, mk in (("mean", lambda: Ridge(alpha=1e9)), ("ridge", lambda: Ridge(alpha=1)), ("SVR rbf", lambda: SVR(C=3, epsilon=0.1)), ("SVR linear", lambda: SVR(kernel="linear", C=1, epsilon=0.1))):
        errs = []
        for a, b_ in StratifiedKFold(5, shuffle=True, random_state=0).split(Xr, bins_):
            sc_ = StandardScaler().fit(Xr[a])
            mu_, sd_ = yr[a].mean(), yr[a].std()
            m_ = mk().fit(sc_.transform(Xr[a]), (yr[a] - mu_) / sd_)
            errs.append(np.sqrt(np.mean((m_.predict(sc_.transform(Xr[b_])) * sd_ + mu_ - yr[b_]) ** 2)))
        res[label] = np.mean(errs)
    rows.append((dname, res["mean"], res["ridge"], res["SVR linear"], res["SVR rbf"], res["SVR rbf"] / res["mean"]))
table(rows, ["dataset", "RMSE mean", "RMSE ridge", "RMSE SVR linear", "RMSE SVR rbf", "SVR rbf / mean"])

**Tuning three hyperparameters.** $C$, $\gamma$ and $\varepsilon$ on a log grid; the heat map shows the RMSE for the best
$\varepsilon$ in every $(C, \gamma)$ cell. A non-linear problem rewards a flexible kernel; the surface is flat near the optimum,
so a coarse grid is enough.

In [ ]:
mu, sd = yc_.mean(), yc_.std()
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc_, yc_, test_size=0.25, random_state=0)
Cs = 10.0 ** np.arange(-1, 4)
gs = 10.0 ** np.arange(-3, 1)
es = [0.01, 0.05, 0.1, 0.3]
g = GridSearchCV(make_pipeline(StandardScaler(), SVR()), {"svr__C": Cs, "svr__gamma": gs, "svr__epsilon": es}, cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)
g.fit(Xc_tr, (yc_tr - mu) / sd)
res = -np.asarray(g.cv_results_["mean_test_score"]).reshape(len(Cs), len(gs), len(es)).min(axis=2) * sd
fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(res, origin="lower", cmap="viridis_r")
ax.set_xticks(range(len(gs)), [f"{v:g}" for v in gs])
ax.set_yticks(range(len(Cs)), [f"{v:g}" for v in Cs])
ax.set(xlabel="gamma (log)", ylabel="C (log)", title="cross-validated RMSE (MPa), best epsilon per cell")
plt.colorbar(im)
plt.show()
best = g.best_params_
pred = g.predict(Xc_te) * sd + mu
print("best", {k.split('__')[1]: float(v) for k, v in best.items()}, f"-> test RMSE {np.sqrt(np.mean((pred - yc_te) ** 2)):.2f} MPa, R2 {1 - np.sum((pred - yc_te) ** 2) / np.sum((yc_te - yc_te.mean()) ** 2):.3f}")
n_sv = len(g.best_estimator_[-1].support_)
print(f"support vectors: {n_sv} of {len(yc_tr)} ({n_sv / len(yc_tr):.0%})")
plt.scatter(yc_te, pred, s=10)
plt.plot([yc_te.min(), yc_te.max()], [yc_te.min(), yc_te.max()], "k:")
plt.xlabel("actual strength (MPa)")
plt.ylabel("predicted")
plt.show()

## Summary

* The SVM chooses the separating line with the **largest margin**; only the **support vectors** define it.
* Soft margin = **hinge loss + $L_2$ penalty** ($\lambda = 1/(Cn)$): trained by the QP solver or by `jax.grad` (subgradient).
* A **kernel** replaces the dot product by a similarity: curved boundaries with the same machine; $\gamma$ and $C$ are
  chosen on a log grid with cross-validation, after **scaling**.
* Cost: $O(n^2)$ to $O(n^3)$ and an $n \times n$ Gram matrix; probabilities need Platt scaling; costs need `class_weight`.
* SVR: a tube of width $\varepsilon$ and the $\varepsilon$-insensitive loss, robust to outliers.
* Next: networks that *learn* the features instead of fixing a kernel (`02_neural_networks.ipynb`).